# 06 — Evaluation

**Project:** Ecommerce-LLM-Finetuning
**Goal of this notebook:**
1. Load the fine-tuned model (base + LoRA adapter)
2. Run inference on a sample of the held-out test set
3. Compute BLEU, ROUGE, and perplexity
4. Display Ground Truth vs. Prediction comparison table

Run this after `05_finetune_llm.ipynb`.


In [ ]:
%%capture
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q evaluate rouge_score sacrebleu nltk pandas


In [ ]:
import os, sys, json, logging, random
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional, Union, List

import pandas as pd


def get_logger(name: str = "ecommerce_llm", level: int = logging.INFO) -> logging.Logger:
    logger = logging.getLogger(name)
    logger.setLevel(level)
    if not logger.handlers:
        handler = logging.StreamHandler(sys.stdout)
        formatter = logging.Formatter(
            "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
            datefmt="%H:%M:%S",
        )
        handler.setFormatter(formatter)
        logger.addHandler(handler)
    return logger


logger = get_logger("notebook06")


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import numpy as np
        import torch
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except Exception:
        pass


@dataclass
class Config:
    project_root: Path = field(
        default_factory=lambda: Path(os.environ.get("PROJECT_ROOT", Path.cwd().parent)).resolve()
    )
    seed: int = 42
    hf_dataset_name: str = "bitext/Bitext-customer-support-llm-chatbot-training-dataset"
    hf_dataset_config: Optional[str] = None
    train_split_ratio: float = 0.8
    val_split_ratio: float = 0.1
    test_split_ratio: float = 0.1
    base_model_name: str = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
    max_seq_length: int = 2048
    load_in_4bit: bool = True
    lora_r: int = 16
    lora_alpha: int = 16
    lora_dropout: float = 0.0
    lora_target_modules: tuple = (
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    )
    use_gradient_checkpointing: str = "unsloth"
    output_dir: str = "outputs"
    num_train_epochs: int = 3
    per_device_train_batch_size: int = 2
    per_device_eval_batch_size: int = 2
    gradient_accumulation_steps: int = 4
    learning_rate: float = 2e-4
    lr_scheduler_type: str = "linear"
    warmup_ratio: float = 0.05
    weight_decay: float = 0.01
    logging_steps: int = 10
    eval_strategy: str = "steps"
    eval_steps: int = 50
    save_strategy: str = "steps"
    save_steps: int = 50
    save_total_limit: int = 2
    bf16: bool = True
    fp16: bool = False
    optim: str = "adamw_8bit"
    report_to: str = "none"
    system_prompt: str = (
        "You are a helpful, polite Ecommerce Customer Support Assistant. "
        "You answer questions about orders, shipping, refunds, returns, "
        "payments, coupons, delivery, and account management. "
        "If a question is unrelated to ecommerce customer support, politely "
        "say that you can only help with ecommerce support questions."
    )
    instruction_header: str = "### Instruction:"
    response_header: str = "### Response:"
    eval_num_samples: int = 50
    eval_max_new_tokens: int = 128
    default_temperature: float = 0.7
    default_max_new_tokens: int = 256
    default_top_p: float = 0.9

    def __post_init__(self) -> None:
        set_seed(self.seed)

    @property
    def data_dir(self) -> Path:
        return self.project_root / "data"

    @property
    def raw_data_dir(self) -> Path:
        return self.data_dir / "raw"

    @property
    def processed_data_dir(self) -> Path:
        return self.data_dir / "processed"

    @property
    def sample_data_dir(self) -> Path:
        return self.data_dir / "sample"

    @property
    def models_dir(self) -> Path:
        return self.project_root / "models"

    @property
    def base_model_dir(self) -> Path:
        return self.models_dir / "base_model"

    @property
    def lora_adapter_dir(self) -> Path:
        return self.models_dir / "lora_adapter"

    @property
    def merged_model_dir(self) -> Path:
        return self.models_dir / "merged_model"

    def ensure_dirs(self) -> None:
        for path in (
            self.raw_data_dir,
            self.processed_data_dir,
            self.sample_data_dir,
            self.base_model_dir,
            self.lora_adapter_dir,
            self.merged_model_dir,
        ):
            path.mkdir(parents=True, exist_ok=True)


def load_jsonl(path) -> list[dict]:
    path = Path(path)
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def detect_device() -> str:
    try:
        import torch
        if torch.cuda.is_available():
            name = torch.cuda.get_device_name(0)
            logger.info(f"CUDA GPU detected: {name}")
            return "cuda"
        if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
            logger.info("Apple MPS backend detected.")
            return "mps"
    except ImportError:
        logger.warning("PyTorch not installed yet; defaulting device check to CPU.")
    logger.warning("No GPU detected — falling back to CPU (training will be slow).")
    return "cpu"


def supports_bf16() -> bool:
    try:
        import torch
        return torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    except Exception:
        return False


class PromptFormatter:
    def __init__(self, config: Optional[Config] = None):
        self.config = config or Config()

    def format_training_example(self, instruction: str, response: str) -> str:
        cfg = self.config
        return (
            f"{cfg.instruction_header}\n"
            f"Customer:\n{instruction.strip()}\n\n"
            f"{cfg.response_header}\n"
            f"{response.strip()}"
        )

    def format_prompt_only(self, instruction: str) -> str:
        cfg = self.config
        return f"{cfg.instruction_header}\nCustomer:\n{instruction.strip()}\n\n{cfg.response_header}\n"

    def format_chat_messages(self, instruction: str, response: Optional[str] = None) -> list[dict]:
        messages = [
            {"role": "system", "content": self.config.system_prompt},
            {"role": "user", "content": instruction.strip()},
        ]
        if response is not None:
            messages.append({"role": "assistant", "content": response.strip()})
        return messages


class ModelEvaluator:
    def __init__(self):
        self._rouge = None
        self._bleu = None

    def _load_metrics(self):
        import evaluate

        if self._rouge is None:
            self._rouge = evaluate.load("rouge")
        if self._bleu is None:
            self._bleu = evaluate.load("sacrebleu")

    def compute_rouge(self, predictions: List[str], references: List[str]) -> dict:
        self._load_metrics()
        return self._rouge.compute(predictions=predictions, references=references)

    def compute_bleu(self, predictions: List[str], references: List[str]) -> dict:
        self._load_metrics()
        refs_formatted = [[r] for r in references]
        result = self._bleu.compute(predictions=predictions, references=refs_formatted)
        return result

    def compute_perplexity(self, model, tokenizer, texts: List[str], device: str = "cuda") -> float:
        import math
        import torch

        model.eval()
        losses = []
        with torch.no_grad():
            for text in texts:
                enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=1024).to(device)
                out = model(**enc, labels=enc["input_ids"])
                losses.append(out.loss.item())
        mean_loss = sum(losses) / max(len(losses), 1)
        return math.exp(mean_loss)

    def build_comparison_table(
        self,
        instructions: List[str],
        ground_truths: List[str],
        predictions: List[str],
    ) -> pd.DataFrame:
        return pd.DataFrame(
            {
                "instruction": instructions,
                "ground_truth": ground_truths,
                "prediction": predictions,
            }
        )

    def full_report(
        self,
        instructions: List[str],
        ground_truths: List[str],
        predictions: List[str],
        model=None,
        tokenizer=None,
        full_texts_for_ppl: Optional[List[str]] = None,
        device: str = "cuda",
    ) -> dict:
        logger.info(f"Evaluating on {len(predictions)} samples...")
        rouge_scores = self.compute_rouge(predictions, ground_truths)
        bleu_scores = self.compute_bleu(predictions, ground_truths)

        ppl = None
        if model is not None and tokenizer is not None and full_texts_for_ppl:
            ppl = self.compute_perplexity(model, tokenizer, full_texts_for_ppl, device=device)

        table = self.build_comparison_table(instructions, ground_truths, predictions)

        logger.info(f"ROUGE: {rouge_scores}")
        logger.info(f"BLEU: {bleu_scores.get('score')}")
        if ppl is not None:
            logger.info(f"Perplexity: {ppl:.3f}")

        return {
            "rouge": rouge_scores,
            "bleu": bleu_scores,
            "perplexity": ppl,
            "comparison_table": table,
        }


class EcommerceSupportBot:
    def __init__(self, config: Optional[Config] = None):
        self.config = config or Config()
        self.formatter = PromptFormatter(self.config)
        self.model = None
        self.tokenizer = None
        self._loaded_from: Optional[str] = None

    def load_merged_model(self, model_dir: Optional[Union[str, Path]] = None):
        from transformers import AutoModelForCausalLM, AutoTokenizer
        import torch

        model_dir = str(model_dir or self.config.merged_model_dir)
        logger.info(f"Loading merged model from {model_dir}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_dir)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_dir,
            device_map="auto",
            torch_dtype=torch.bfloat16 if detect_device() == "cuda" else torch.float32,
        )
        self._loaded_from = "merged"
        return self.model, self.tokenizer

    def load_adapter_model(
        self,
        base_model_name: Optional[str] = None,
        adapter_dir: Optional[Union[str, Path]] = None,
    ):
        from unsloth import FastLanguageModel

        cfg = self.config
        base_model_name = base_model_name or cfg.base_model_name
        adapter_dir = str(adapter_dir or cfg.lora_adapter_dir)

        logger.info(f"Loading base model '{base_model_name}' + adapter '{adapter_dir}'")
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=base_model_name,
            max_seq_length=cfg.max_seq_length,
            load_in_4bit=cfg.load_in_4bit,
        )
        model.load_adapter(adapter_dir)
        FastLanguageModel.for_inference(model)

        self.model, self.tokenizer = model, tokenizer
        self._loaded_from = "adapter"
        return model, tokenizer

    def merge_and_save(self, save_dir: Optional[Union[str, Path]] = None) -> Path:
        if self.model is None:
            raise RuntimeError("Load a base+adapter model first via load_adapter_model().")
        save_dir = Path(save_dir or self.config.merged_model_dir)
        save_dir.mkdir(parents=True, exist_ok=True)
        logger.info(f"Merging LoRA weights and saving to {save_dir} ...")
        merged = self.model.merge_and_unload() if hasattr(self.model, "merge_and_unload") else self.model
        merged.save_pretrained(str(save_dir))
        self.tokenizer.save_pretrained(str(save_dir))
        logger.info("Merge complete.")
        return save_dir

    def generate(
        self,
        instruction: str,
        max_new_tokens: Optional[int] = None,
        temperature: Optional[float] = None,
        top_p: Optional[float] = None,
        use_chat_template: bool = True,
    ) -> str:
        import torch

        if self.model is None or self.tokenizer is None:
            raise RuntimeError("No model loaded. Call load_merged_model() or load_adapter_model().")

        cfg = self.config
        max_new_tokens = max_new_tokens or cfg.default_max_new_tokens
        temperature = temperature if temperature is not None else cfg.default_temperature
        top_p = top_p if top_p is not None else cfg.default_top_p

        if use_chat_template and getattr(self.tokenizer, "chat_template", None):
            messages = self.formatter.format_chat_messages(instruction)
            input_ids = self.tokenizer.apply_chat_template(
                messages, add_generation_prompt=True, return_tensors="pt"
            ).to(self.model.device)
            attention_mask = torch.ones_like(input_ids)
        else:
            prompt = self.formatter.format_prompt_only(instruction)
            enc = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
            input_ids, attention_mask = enc["input_ids"], enc["attention_mask"]

        with torch.no_grad():
            output_ids = self.model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=max_new_tokens,
                temperature=max(temperature, 1e-3),
                top_p=top_p,
                do_sample=temperature > 0,
                pad_token_id=self.tokenizer.eos_token_id,
            )

        new_tokens = output_ids[0][input_ids.shape[-1] :]
        response = self.tokenizer.decode(new_tokens, skip_special_tokens=True)
        return response.strip

    def chat(self, history: list[tuple[str, str]], new_message: str, **gen_kwargs) -> str:
        return self.generate(new_message, **gen_kwargs)


cfg = Config()
device = detect_device()


## 1. Load fine-tuned model (base + LoRA adapter)

In [ ]:
bot = EcommerceSupportBot(cfg)
model, tokenizer = bot.load_adapter_model()
print("Model loaded from adapter:", cfg.lora_adapter_dir)


## 2. Load test set and sample

In [ ]:
test_records = load_jsonl(cfg.processed_data_dir / "test.jsonl")
test_df = pd.DataFrame(test_records)

n_eval = min(cfg.eval_num_samples, len(test_df))
eval_df = test_df.sample(n_eval, random_state=cfg.seed).reset_index(drop=True)
print(f"Evaluating on {len(eval_df)} held-out test samples")
eval_df.head(3)


## 3. Generate predictions

In [ ]:
from tqdm.auto import tqdm

predictions = []
for _, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
    pred = bot.generate(
        row["instruction"],
        max_new_tokens=cfg.eval_max_new_tokens,
        temperature=0.3,   # lower temperature for more deterministic eval
        use_chat_template=False,  # match the raw instruction format used in training
    )
    predictions.append(pred)

eval_df["prediction"] = predictions
eval_df.head(3)


## 4. Compute BLEU, ROUGE, and Perplexity

In [ ]:
evaluator = ModelEvaluator()

report = evaluator.full_report(
    instructions=eval_df["instruction"].tolist(),
    ground_truths=eval_df["response"].tolist(),
    predictions=eval_df["prediction"].tolist(),
    model=model,
    tokenizer=tokenizer,
    full_texts_for_ppl=eval_df["text"].tolist() if "text" in eval_df.columns else None,
    device=device,
)

print("ROUGE:", report["rouge"])
print("BLEU:", report["bleu"]["score"])
print("Perplexity:", report["perplexity"])


## 5. Ground Truth vs. Prediction comparison table

In [ ]:
comparison_table = report["comparison_table"]
pd.set_option("display.max_colwidth", 200)
comparison_table.head(10)


In [ ]:
# Save the full comparison table + metrics for the README / portfolio writeup
out_dir = cfg.project_root / "data" / "processed"
comparison_table.to_csv(out_dir / "eval_comparison_table.csv", index=False)

metrics_summary = {
    "rouge1": report["rouge"].get("rouge1"),
    "rouge2": report["rouge"].get("rouge2"),
    "rougeL": report["rouge"].get("rougeL"),
    "bleu": report["bleu"].get("score"),
    "perplexity": report["perplexity"],
    "n_eval_samples": len(eval_df),
}
import json
with open(out_dir / "eval_metrics.json", "w") as f:
    json.dump(metrics_summary, f, indent=2)

print(metrics_summary)


## 6. Manual qualitative spot-check

Print a few full examples side-by-side for a closer read.

In [ ]:
for i in range(min(5, len(eval_df))):
    row = eval_df.iloc[i]
    print("=" * 80)
    print("INSTRUCTION:", row["instruction"])
    print("-" * 80)
    print("GROUND TRUTH:", row["response"])
    print("-" * 80)
    print("PREDICTION:  ", row["prediction"])
print("=" * 80)


## Summary

- Generated predictions for a sample of the held-out test set
- Computed BLEU, ROUGE-1/2/L, and perplexity
- Saved a full comparison table and metrics summary to `data/processed/`

Next: `07_inference.ipynb` to merge the adapter into a standalone model and build the final chat interface.